# Rețele neuronale și deep learning — Seminarul 10: Învățarea prin întărire: tranzacționare, acoperirea riscului și dispecerizarea bateriilor
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L10_seminar_ro.ipynb)

Sarcinile seminarului, pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Scrieți codul în celulele goale de sub fiecare sarcină; păstrați valoarea fixă a generatorului aleator, ca rezultatele să poată fi comparate.

In [ ]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Pregătire: prețurile și modelul bateriei

In [ ]:
import torch, torch.nn as nn
x = pd.read_csv(DATA + 'power_da_prices_hourly_utc.csv', parse_dates=['timestamp_utc']).set_index('timestamp_utc')
x.index = x.index.tz_convert('Europe/Brussels')
END = '2026-10-31'                                  # last complete month of the course data
x = x.loc[:END + ' 23:59']
x['day'] = x.index.tz_localize(None).normalize(); x['hour'] = x.index.hour

def day_matrix(zone):
    # delivery days (CET) x 24 hours; only days with 24 reported hours
    d = x[['day', 'hour', zone]].dropna()
    n = d.groupby('day')[zone].count()
    return d[d.day.isin(n[n == 24].index)].pivot_table(index='day', columns='hour', values=zone).sort_index()

RO = day_matrix('RO')
print(RO.index.min().date(), RO.index.max().date(), len(RO), 'complete days')

In [ ]:
E_MAX, ETA = 2, 0.85                 # 2 MWh, 1 MWh of stored energy per hour, 85 % round-trip efficiency
ETA_C = ETA_D = np.sqrt(ETA)
ACTS = np.array([-1, 0, 1])          # charge, idle, discharge

def reward(p, a):
    return np.where(a > 0, ETA_D * p, np.where(a < 0, -p / ETA_C, 0.0))

def dp_days(P):
    # backward induction V_h(s) = max_a { r(p_h, a) + V_{h+1}(s - a) }, all days at once
    D = P.shape[0]
    V = np.zeros((D, 25, E_MAX + 1)); A = np.zeros((D, 24, E_MAX + 1), dtype=int)
    for h in range(23, -1, -1):
        best = np.full((D, E_MAX + 1), -np.inf); arg = np.zeros((D, E_MAX + 1), dtype=int)
        for s in range(E_MAX + 1):
            for a in ACTS:
                if 0 <= s - a <= E_MAX:
                    q = reward(P[:, h], a) + V[:, h + 1, s - a]
                    b = q > best[:, s]; best[b, s] = q[b]; arg[b, s] = a
        V[:, h] = best; A[:, h] = arg
    return V, A

**Sarcina 1.** Calculați venitul bateriei cu previziune perfectă, pe ani, pentru Ungaria (HU) și Germania-Luxemburg (DE_LU) și comparați-l cu România. Care piață a plătit cel mai mult în 2025?

In [ ]:
# codul vostru aici

**Sarcina 2.** Rulați din nou programarea dinamică cu un randament al ciclului de 0,90 și 0,75. Cît se modifică venitul din 2025 în România? Explicați de ce randamentul contează mai mult în zilele cu o diferență mică de preț.

In [ ]:
# codul vostru aici

**Sarcina 3.** Reperul naiv: aplicați planul optim de ieri la prețurile de azi, în 2025-2026. Ce parte din venitul cu previziune perfectă obține?

In [ ]:
# codul vostru aici

**Sarcina 4.** O baterie de o oră: setați E_MAX = 1 și recalculați venitul cu previziune perfectă din 2025. Valorează a doua oră de stocare jumătate din venit?

In [ ]:
# codul vostru aici

## Tranzacționare și acoperirea riscului

In [ ]:
m = pd.read_csv(DATA + 'markets_daily.csv', parse_dates=['date'])
w = m.pivot(index='date', columns='ticker', values='close').loc[:END]
px = w['SP500'].dropna(); vix = w['VIX'].reindex(px.index).ffill()
r = np.log(px).diff().shift(-1).dropna()
ma = (px / px.rolling(200).mean() - 1).reindex(r.index)

**Sarcina 5.** Evaluați regula mediei mobile de 200 de zile pe 2020 - octombrie 2026, cu costuri de 0, 5, 20 și 50 de puncte de bază pe tranzacție, și comparați raportul Sharpe cu strategia cumpără și păstrează.

In [ ]:
# codul vostru aici

**Sarcina 6.** Acoperiți opțiunea call la bani pe o lună cu delta calculat din volatilitatea realizată pe 21 de zile în loc de VIX. Comparați abaterea standard a rezultatului acoperirii cu cea a acoperirii bazate pe VIX (fără costuri).

In [ ]:
# codul vostru aici